# Industrial Equipment Failure Detection using LSTM Autoencoder

This cleaned portfolio notebook reproduces the project workflow through the modular `src/` package. The included dataset is synthetic and this work is not intended for operational or safety decisions.


In [ ]:
from pathlib import Path
import sys
import json
import pandas as pd
import matplotlib.pyplot as plt

PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.insert(0, str(PROJECT_ROOT))


## 1. Generate and inspect the deterministic sensor dataset


In [ ]:
from src.synthetic_data import generate_predictive_maintenance_data
from src.feature_engineering import sensor_summary

df, sensor_cols = generate_predictive_maintenance_data()
display(df.head())
display(sensor_summary(df, sensor_cols))


## 2. Leakage-resistant preprocessing and unit split


In [ ]:
from src.data_preprocessing import (
    DatasetSchema, clean_sensor_data, split_by_unit,
    fit_training_scaler, apply_scaler,
)

schema = DatasetSchema(sensor_cols=sensor_cols)
clean = clean_sensor_data(df, schema)
train_df, val_df, test_df = split_by_unit(clean, schema)
scaler = fit_training_scaler(train_df, sensor_cols)
train_scaled = apply_scaler(train_df, sensor_cols, scaler)
val_scaled = apply_scaler(val_df, sensor_cols, scaler)
test_scaled = apply_scaler(test_df, sensor_cols, scaler)
print(train_df.shape, val_df.shape, test_df.shape)


## 3. Build 20-step multivariate windows


In [ ]:
from src.sequence_generation import build_sequences

train_batch = build_sequences(train_scaled, schema, window_size=20)
val_batch = build_sequences(val_scaled, schema, window_size=20)
test_batch = build_sequences(test_scaled, schema, window_size=20)
X_train = train_batch.sequences[train_batch.labels == 0]
X_val_healthy = val_batch.sequences[val_batch.labels == 0]
print(X_train.shape, X_val_healthy.shape, test_batch.sequences.shape)


## 4. Load the supplied trained model through the portable inference pipeline


In [ ]:
from src.inference_pipeline import PredictiveMaintenancePipeline

pipeline = PredictiveMaintenancePipeline.from_artifacts(PROJECT_ROOT / "models")
result = pipeline.score_dataframe(test_df)
display(result.predictions.head())
print("Backend:", result.backend_name)


## 5. Verify labeled held-out metrics


In [ ]:
from src.model_evaluation import evaluate_labeled_scores

evaluation = evaluate_labeled_scores(
    result.predictions["true_label"].to_numpy(),
    result.predictions["reconstruction_error"].to_numpy(),
    float(pipeline.metadata["threshold"]),
)
print(json.dumps(evaluation, indent=2))


## 6. Inspect one equipment health timeline


In [ ]:
unit_result = pipeline.score_dataframe(test_df, selected_unit=105)
display(unit_result.predictions.tail(15))
unit_result.predictions.plot(
    x="window_end", y="reconstruction_error", figsize=(12, 4),
    title="Equipment 105 reconstruction error"
)
plt.axhline(pipeline.metadata["threshold"], linestyle="--")
plt.show()


## 7. Optional retraining


In [ ]:
# Requires: pip install -r requirements-dev.txt
# from src.model_training import build_lstm_autoencoder, train_autoencoder
# model = build_lstm_autoencoder(20, len(sensor_cols))
# history = train_autoencoder(model, X_train, X_val_healthy)


## Conclusion

The artifact demonstrates a complete anomaly-detection workflow, while the audit documents important limitations: synthetic data, global threshold sensitivity, false-positive cost, and the difference between anomaly detection and confirmed failure diagnosis.
